<h3 style="color: #2E86C1; border-bottom: 2px solid #2E86C1; padding-bottom: 5px;">
  📥 1. Extracción de Datos (Extract)
</h3>
<p>
  Iniciamos el proceso importando las librerías necesarias y cargando el dataset original proporcionado por la aseguradora. Verificamos su correcta lectura visualizando los primeros registros.
</p>

In [22]:
import pandas as pd

df = pd.read_csv('train.csv')

df.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


<h3 style="color: #2E86C1; border-bottom: 2px solid #2E86C1; padding-bottom: 5px;">
  📊 2. Auditoría y Análisis Estadístico Básico
</h3>
<p>
  Realizamos una validación de integridad buscando registros duplicados y ejecutamos un resumen estadístico para revisar las distribuciones numéricas, confirmando la ausencia de errores de carga.
</p>

In [23]:
df.duplicated().sum()

np.int64(0)

In [24]:
# Verificamos la cantidad de valores nulos por columna
df.isnull().sum()

id                      0
Gender                  0
Age                     0
Driving_License         0
Region_Code             0
Previously_Insured      0
Vehicle_Age             0
Vehicle_Damage          0
Annual_Premium          0
Policy_Sales_Channel    0
Vintage                 0
Response                0
dtype: int64

In [25]:
df.describe()

,id,Age,Driving_License,Region_Code,Previously_Insured,Annual_Premium,Policy_Sales_Channel,Vintage,Response
count,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000
mean,190555.000000,38.822584,0.997869,26.388807,0.458210,30564.389581,112.034295,154.347397,0.122563
std,110016.836208,15.511611,0.046110,13.229888,0.498251,17213.155057,54.203995,83.671304,0.327936
min,1.000000,20.000000,0.000000,0.000000,0.000000,2630.000000,1.000000,10.000000,0.000000
25%,95278.000000,25.000000,1.000000,15.000000,0.000000,24405.000000,29.000000,82.000000,0.000000
50%,190555.000000,36.000000,1.000000,28.000000,0.000000,31669.000000,133.000000,154.000000,0.000000
75%,285832.000000,49.000000,1.000000,35.000000,1.000000,39400.000000,152.000000,227.000000,0.000000
max,381109.000000,85.000000,1.000000,52.000000,1.000000,540165.000000,163.000000,299.000000,1.000000


<h3 style="color: #2E86C1; border-bottom: 2px solid #2E86C1; padding-bottom: 5px;">
  📝 3. Estandarización de Variables de Texto
</h3>
<p>
  Convertimos las columnas categóricas a minúsculas para evitar discrepancias de formato (ej. "Male" vs "male") que puedan afectar la agrupación de datos.
</p>

In [26]:
df['Gender'] = df['Gender'].str.lower()
df['Vehicle_Age'] = df['Vehicle_Age'].str.lower()
df['Vehicle_Damage'] = df['Vehicle_Damage'].str.lower()

df.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,male,44,1,28.0,0,> 2 years,yes,40454.0,26.0,217,1
1,2,male,76,1,3.0,0,1-2 year,no,33536.0,26.0,183,0
2,3,male,47,1,28.0,0,> 2 years,yes,38294.0,26.0,27,1
3,4,male,21,1,11.0,1,< 1 year,no,28619.0,152.0,203,0
4,5,female,29,1,41.0,1,< 1 year,no,27496.0,152.0,39,0


<h3 style="color: #2E86C1; border-bottom: 2px solid #2E86C1; padding-bottom: 5px;">
  🧹 4. Tratamiento de Valores Atípicos (Outliers)
</h3>
<p>
  Para evitar que primas anuales extremas distorsionen los promedios y gráficos posteriores, filtramos y removemos los registros que superan los $500,000, manteniendo la integridad del análisis sobre el comportamiento general.
</p>

In [27]:
outliers_premium = df[df['Annual_Premium'] > 500000]

outliers_premium

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
11319,11320,female,50,1,46.0,1,1-2 year,no,508073.0,26.0,192,0
54743,54744,male,26,1,28.0,0,< 1 year,yes,540165.0,156.0,245,1
144282,144283,female,53,1,28.0,1,1-2 year,no,540165.0,26.0,134,0
190154,190155,male,47,1,28.0,0,1-2 year,yes,540165.0,42.0,24,0
268332,268333,male,46,1,28.0,0,1-2 year,yes,540165.0,124.0,59,0


In [28]:
df = df[df['Annual_Premium'] <= 500000]

print(f"Total de registros tras limpieza: {len(df)}")

Total de registros tras limpieza: 381104


<h3 style="color: #2E86C1; border-bottom: 2px solid #2E86C1; padding-bottom: 5px;">
  ⚙️ 5. Feature Engineering: Segmentación por Edades
</h3>
<p>
  Transformamos la variable continua <code>Age</code> en una variable categórica de rangos. Agrupar a los clientes nos permitirá identificar tendencias de compra intergeneracionales con mayor claridad.
</p>

In [29]:
cortes = [19, 30, 40, 50, 60, 100]
etiquetas = ['20-30 años', '31-40 años', '41-50 años', '51-60 años', '60+ años']

df['Age_Group'] = pd.cut(df['Age'], bins=cortes, labels=etiquetas)

df.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response,Age_Group
0,1,male,44,1,28.0,0,> 2 years,yes,40454.0,26.0,217,1,41-50 años
1,2,male,76,1,3.0,0,1-2 year,no,33536.0,26.0,183,0,60+ años
2,3,male,47,1,28.0,0,> 2 years,yes,38294.0,26.0,27,1,41-50 años
3,4,male,21,1,11.0,1,< 1 year,no,28619.0,152.0,203,0,20-30 años
4,5,female,29,1,41.0,1,< 1 year,no,27496.0,152.0,39,0,20-30 años


<h3 style="color: #2E86C1; border-bottom: 2px solid #2E86C1; padding-bottom: 5px;">
  ⚙️ 6. Feature Engineering Avanzado: Indicador de Alto Riesgo
</h3>
<p>
  Creamos una nueva variable binaria llamada <code>Alto_Riesgo</code> que combina los antecedentes de daños en el vehículo (<code>yes</code>) con la ausencia de póliza previa (<code>0</code>), identificando al segmento de clientes propicio para la venta cruzada.
</p>

In [30]:
df['Alto_Riesgo'] = ((df['Vehicle_Damage'] == 'yes') & (df['Previously_Insured'] == 0)).astype(int)

df['Alto_Riesgo'].value_counts()

Alto_Riesgo
0    198616
1    182488
Name: count, dtype: int64

<h3 style="color: #2E86C1; border-bottom: 2px solid #2E86C1; padding-bottom: 5px;">
  💾 7. Carga (Load): Exportación del Dataset Limpio
</h3>
<p>
  Con los datos auditados, estandarizados y enriquecidos, exportamos el resultado a un nuevo archivo <code>train_clean.csv</code> para consumirlo en la fase de Análisis Exploratorio (EDA).
</p>

In [31]:
df.to_csv('train_clean.csv', index=False)